# 9.1. Working with Sequences
D2L의 Working with Sequences장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [ ]:
%matplotlib inline

import matplotlib
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

print(matplotlib.get_backend())

## 1. 시퀀스 데이터란?

지금까지는 보통 하나의 입력을 하나의 벡터로 생각했다.

$$
\mathbf{x} \in \mathbb{R}^d
$$

예를 들어서 이미지 분류에서는 하나의 이미지가 하나의 입력이었다. 하지만 시퀀스 데이터는 입력이 여러 시점에 걸쳐 순서대로 존재한다.

$$
\mathbf{x}_1,\mathbf{x}_2,\ldots,\mathbf{x}_T
$$

여기에서 $t$는 시간 또는 순서를 의미한다.

대표적인 시퀀스 데이터는 이렇다. 시퀀스에서는 데이터의 순서가 중요하다.

- 주가: 오늘 가격은 어제 가격과 관련됨
- 문장: 다음 단어는 앞 단어들과 관련됨
- 센서 데이터: 현재 측정값은 이전 측정값과 관련됨
- 환자 기록: 오늘 상태는 이전 치료 과정과 관련됨

## 2. 일반 데이터와 시퀀스 데이터의 차이

일반적인 지도학습에서는 각 데이터가 서로 독립이라고 가정하는 경우가 많다.

하지만 시퀀스에서는

$$
x_1,x_2,x_3,\ldots
$$

가 서로 영향을 주기 때문에 독립이라고 보기 어렵다.

예를 들어서 나는 오늘 학교에 _____ 에서 다음 단어가 간다, 갔다, 왔다 등이 될 가능성은 앞의 단어들에 의해 결정된다.

시퀀스 모델의 핵심은 현재/미래 값을 예측할 때 과거 정보를 이용한다는 것이다.

## 3. 시퀀스 문제의 여러 형태

시퀀스를 이용하는 문제는 여러 행태가 존재한다.

### Sequence -> Fixed

영화 리뷰 문장 -> 긍정 / 부정

문장은 시퀀스지만 출력은 하나다.

### Fixed -> Sequence

이미지 -> "강아지가 잔디 위에서 뛰고 있다"

이미지 하나를 입력받아 문장을 생성할 수 있다.

### Sequence -> Sequence

I love deep learning -> 나는 딥러닝을 좋아한다

입력도 시퀀스이고 출력도 시퀀스다. 이후 RNN, LSTM, Transformer 등은 이런 시퀀스 문제를 처리하기 위해 사용된다.

## 4. 자기회귀 모델 (Autoregressive Model)

시계열에서 현재 값 $x_t$를 예측한다고 생각해보자. 현재 값을 예측하기 위해 과거 값을 사용할 수 있다.

$$
P(x_t \mid x_{t-1},x_{t-2},\ldots,x_1)
$$

    과거 데이터 -> 현재 데이터 예측

자기 자신의 과거 값을 이용해서 자신의 미래 값을 예측하기 때문에 이를 자기회귀(Autoregressive) 모델이라고 한다.

예를 들어 주가를 예측한다면 이런 구조이다.

```text
월요일 100
화요일 103
수요일 105
목요일 104
↓
금요일 가격 예측
```

## 5. 모든 과거를 사용하면 생기는 문제

이론적으로는 $x_t$를 예측할 때 모든 과거 값을 사용할 수 있다.

$$
x_1,x_2,\ldots,x_{t-1}
$$

하지만 문제가 있다. $t$가 증가할수록 입력의 길이도 계속 증가한다.

    x5 예측 -> x1 x2 x3 x4
    x100 예측 -> x1 x2 ... x99

일반적인 Linear Layer나 MLP는 입력 크기가 고정되어 있어야 하기 때문에 이런 구조를 바로 처리하기 어렵다. 이를 해결하는 대표적인 방법이 두 가지다.

### 방법 1: 최근 일정 구간만 사용

최근 $\tau$개의 값만 사용한다.

$$
x_{t-\tau},\ldots,x_{t-2},x_{t-1}
$$

### 방법 2: 과거 정보를 하나의 상태로 압축

$$
h_t=g(h_{t-1},x_{t-1})
$$

여기서 $h_t$는 지금까지의 과거 정보를 요약한 값이다. 이 아이디어가 이후 배우는 RNN의 hidden state와 연결된다.

## 6. 고정 길이 Window 사용하기

예를 들어서

$$
\tau=4
$$

로 설정하면 현재 값 하나를 예측하기 위해 바로 이전 값 4개만 사용한다.

```text
[x1, x2, x3, x4] → x5
[x2, x3, x4, x5] → x6
[x3, x4, x5, x6] → x7
```

이제 모든 입력의 길이가 4로 동일해진다.

    가변 길이 과거 데이터 -> 고정 길이 Window -> 일반적인 신경망에서도 학습 가능

이 방식을 Sliding Window라고 생각하면 된다.

## 7. 시퀀스 모델과 확률

언어 모델에서는 문장 전체가 등장할 확률을 생각할 수 있다.

문장이 이렇게 구성되어 있다면

$$
x_1,x_2,\ldots,x_T
$$

$$
P(x_1,x_2,\ldots,x_T)
$$

이것을 구하고 싶다. 확률의 Chain Rule을 사용하면 다음처럼 나눌 수 있다.
$$
P(x_1)
\prod_{t=2}^{T}
P(x_t \mid x_1,\ldots,x_{t-1})
$$

    첫 번째 단어가 나올 확률 x 앞 단어를 봤을 때 두 번째 단어가 나올 확률 x 앞 단어들을 봤을 때 세 번째 단어가 나올 확률 x ...

예를 들어 `나는 학교에 간다` 라면 개념적으로

$$
P(\text{나는})
\times
P(\text{학교에}\mid\text{나는})
\times
P(\text{간다}\mid\text{나는, 학교에})
$$

이렇게 볼 수 있다. 이게 현대 언어 모델의 기본적인 아이디어와 연결된다.

## 8. 마르코프 가정

모든 과거 정보를 반드시 사용할 필요가 없다고 가정할 수도 있다.

`아주 오래된 과거는 무시하고 최근 정보만으로 미래를 예측하자.` 라는 것이다. $\tau$개의 최근 값만 사용하면 이렇다.

$$
P(x_t \mid x_{t-1},\ldots,x_{t-\tau})
$$

특히 $\tau=1$이면

$$
P(x_t\mid x_{t-1})
$$

바로 이전 상태만 보고 다음 상태를 결정한다. 이를 1차 Markov Model이라고 한다.

전체 시퀀스 확률도 이렇게 단순화할 수 있다.

$$
P(x_1)
\prod_{t=2}^{T}
P(x_t\mid x_{t-1})
$$

실제 문장에서는 훨씬 이전의 단어도 영향을 줄 수 있기 때문에 완벽한 마르코프 가정은 아니지만, 계산을 단순하게 만들기 위해 이런 근사를 사용할 수 있다.

## 9. 실습용 시계열 데이터 만들기

sin 함수를 이용해 시계열 데이터를 만든다.

$$
x_t=\sin(0.01t)+\epsilon
$$

여기서 $\epsilon$은 랜덤 노이즈다.

완벽한 sin 곡선이 아니라 약간의 노이즈가 섞인 시계열이 만들어진다.

In [ ]:
T = 1000

time = torch.arange(1, T + 1, dtype=torch.float32)

x = torch.sin(0.01 * time) + torch.randn(T) * 0.2

plt.figure(figsize=(8, 3))
plt.plot(time, x)
plt.xlabel("time")
plt.ylabel("x")
plt.show()

## 10. 시계열을 학습 데이터로 변환하기

최근 4개 값으로 $\tau=4$을 예측한다고 해보자. 그러면 학습 데이터는 다음과 같이 만들어진다.

```text
입력                  정답

[x1, x2, x3, x4]  →   x5
[x2, x3, x4, x5]  →   x6
[x3, x4, x5, x6]  →   x7
...
```



In [ ]:
tau = 4

features = []

for i in range(tau):
    features.append(x[i:T - tau + i])

features = torch.stack(features, dim=1)

labels = x[tau:].reshape(-1, 1)

print(features.shape) # [996, 4] 
print(labels.shape)   # [996, 1] 
# 하나의 데이터는 4개의 값 -> 다음 값 1개이다.
print(features[0])
print(labels[0])

## 11. 간단한 자기회귀 모델 학습

아직 RNN을 사용하지 않아도 된다.

최근 4개의 값을 입력받는 간단한 Linear Layer만으로도 다음 값을 예측할 수 있다.

In [ ]:
from torch import nn

num_train = 600

model = nn.Linear(tau, 1)

loss_fn = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

X_train = features[:num_train]
y_train = labels[:num_train]

for epoch in range(100):

    pred = model(X_train)

    loss = loss_fn(pred, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(
            f"epoch={epoch+1}, "
            f"loss={loss.item():.4f}"
        )

여기서 모델이 배우는 것은 결국 이거다.
$$
w_1x_{t-4}
+w_2x_{t-3}
+w_3x_{t-2}
+w_4x_{t-1}
+b
$$

과거 4개의 값에 각각 얼마나 중요도를 줄지 $w$를 학습한다.

## 12. 1-step 예측과 Multi-step 예측

### 1-step Prediction
실제 과거 데이터가 주어진 상태에서 바로 다음 값을 예측한다.

    [x601, x602, x603, x604] -> x605 예측

이 경우 모델은 비교적 정확하게 예측할 수 있다.
```py
with torch.no_grad():
    one_step_preds = model(features).squeeze()

plt.figure(figsize=(8, 3))

plt.plot(
    time[tau:],
    labels.squeeze(),
    label="true"
)

plt.plot(
    time[tau:],
    one_step_preds,
    label="1-step prediction"
)

plt.legend()
plt.show()
```

하지만 여러 스텝 이후를 예측하려면 문제가 생긴다. 예를 들어 $x_{604}$까지만 알고 있다고 하자.

$$ 
f(x_{601},x_{602},x_{603},x_{604})
$$

까지는 실제 데이터를 사용할 수 있다. 하지만 $x_{606}$을 예측할 때는 실제 $x_{605}$를 모른다.

그래서
$$
f(x_{602},x_{603},x_{604},\hat{x}_{605})
$$

처럼 자신의 예측값을 다시 입력으로 사용해야 한다.
$$
f(x_{603},x_{604},\hat{x}{605},\hat{x}{606})
$$

    예측 -> 그 예측을 다시 입력 -> 다음 예측 -> 그 예측을 다시 입력 -> 다음 예측 -> ...

이렇게 반복된다. 이것를 Multi-step Prediction이라고 한다. 문제는 이전 예측에서 발생한 작은 오차가 다음 입력으로 들어간다는 것이다.

```text
1-step
작은 오차

↓

4-step
오차 증가

↓

16-step
더 큰 오차

↓

64-step
예측 붕괴
```

따라서 일반적으로 미래를 멀리 예측할수록 정확도가 급격히 떨어진다.

날씨 예측에서 내일의 날씨는 비교적 잘 맞지만 몇 주 뒤의 날씨는 정확하게 예측하기 어려운 것과 비슷하다.

## 13. 오늘의 정리

- 시퀀스 데이터는 순서가 존재하는 데이터다.
- 시퀀스에서는 현재 값이 이전 값들과 관련되어 있기 때문에 일반적인 독립 데이터와 다르게 처리해야 한다.
- 자기회귀 모델은 과거의 자기 자신을 이용하여 미래의 자기 자신을 예측하는 모델이다.
- 모든 과거를 입력하면 입력 크기가 계속 증가하기 때문에 최근 $\tau$개만 사용하는 Window 방식을 사용할 수 있다.
- $\tau$개의 최근 값만으로 미래를 예측할 수 있다고 가정하는 것이 Markov 관점이다.
- 언어 모델도 결국 앞의 단어들을 이용하여 다음 단어의 확률을 예측하는 자기회귀 문제로 볼 수 있다.
- Hidden State $h_t$를 사용하면 과거 전체를 직접 저장하는 대신 과거 정보를 하나의 상태로 압축할 수 있으며, 이 아이디어가 RNN으로 이어진다.
- 시계열 데이터는 [과거 값들] → [다음 값] 형태의 학습 데이터로 변환할 수 있다.
- 1-step prediction은 실제 과거 데이터를 사용하기 때문에 비교적 쉽다.
- Multi-step prediction은 자신의 이전 예측값을 다시 입력으로 사용하기 때문에 오차가 누적된다.
- 시계열 학습에서는 미래 데이터를 과거 예측에 사용하면 안 된다.
- 이번 장의 핵심은 RNN 구현 자체보다 왜 시퀀스를 위한 특별한 모델이 필요한지 이해하는 것이다.